# <center> Some plots from the DEA results

## <center> Description

- env: squidpy_v06

## <center> Set-up

#### **libs**

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
import sys
import math
from pathlib import Path
from scipy import sparse

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plotting
# import marsilea as ma
# import marsilea.plotter as mp
# from matplotlib.colors import NoNorm # for marsilea dotplot
import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    ## input
    in_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    in_deg = ROOT / "output/DEA/objects/calibrate_pvalues/deg_calibrated.tsv"
    ## output
    ### plots
    out_plots = ROOT / "output/DEA/plots/plot_DEA"
    out_plots.mkdir(parents=True, exist_ok=True)


## <center> Main

#### **Load data**

In [3]:
# adata

adata = sc.read_h5ad(in_adata)
## remove group B
adata = adata[adata.obs['QC_group']=='A'].copy()
## make sure counts are norm
adata.X = adata.layers['cts.deseq2_norm'].copy()
## col with ENSEMBL ID
adata.var['ENS']=adata.var.index
## rename var_names
adata.var_names = [symb if (not pd.isna(symb)) else ENS for ENS,symb in zip(adata.var.index, adata.var['genesymbol'])]
## because some genesymbol won't be unique
adata.var_names_make_unique()
## conv dictionary to re-name degs
conv_dict = {ENS:new_symb for ENS,new_symb in zip(adata.var['ENS'],adata.var.index)}

adata

AnnData object with n_obs × n_vars = 50 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ENS'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [4]:
# add individualID

col_dict = {
    'SD1': 'individual_01','SD2': 'individual_01','SD3': 'individual_02','SD4': 'individual_02',
    'SD5': 'individual_03','SD6': 'individual_03','SD7': 'individual_04','SD8': 'individual_04',
    'SD9': 'individual_05','SD10': 'individual_05','SD11': 'individual_06','SD12': 'individual_06',
    'SD13': 'individual_07','SD14': 'individual_07','SD15': 'individual_08','SD16': 'individual_08',
    'SD17': 'individual_09','SD18': 'individual_09','SD19': 'individual_10','SD20': 'individual_10',
    'SD21': 'individual_11','SD22': 'individual_11','SD23': 'individual_12','SD24': 'individual_12',
    'SD25': 'individual_13','SD26': 'individual_13','SD27': 'individual_14','SD28': 'individual_14',
    'SD29': 'individual_15','SD30': 'individual_15','SD31': 'individual_16','SD32': 'individual_16',
    'SD33': 'individual_17','SD34': 'individual_17','SD35': 'individual_18','SD36': 'individual_18',
    'SD37': 'individual_19','SD38': 'individual_19','SD39': 'individual_20','SD40': 'individual_20',
    'SD41': 'individual_21','SD42': 'individual_21','SD43': 'individual_22','SD44': 'individual_22',
    'SD45': 'individual_23','SD46': 'individual_23','SD47': 'individual_24','SD48': 'individual_24',
    'SD49': 'individual_25','SD50': 'individual_25','SD51': 'individual_26','SD52': 'individual_26',
    'SD53': 'individual_27','SD54': 'individual_27','SD55': 'individual_28','SD56': 'individual_28',
    'SD57': 'individual_29','SD58': 'individual_29','SD59': 'individual_30','SD60': 'individual_30',
    'SD61': 'individual_31','SD62': 'individual_31','SD63': 'individual_32','SD64': 'individual_32',
    'SD65': 'individual_33','SD66': 'individual_33',
}
adata.obs['individualID']=adata.obs.index.map(col_dict)
adata.obs['individualID']=adata.obs['individualID'].astype('category')

In [5]:
# DEA results

deg = pd.read_csv(in_deg,sep='\t',index_col=0).set_index('gene_id')
## rename genesymbol as in adata.var
deg['genesymbol'] = deg.index.map(conv_dict)

deg.head()

,genesymbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,significant_pvalue_0.05,significant_padj_0.05,direction
gene_id,,,,,,,,,,,
ENSG00000283293,RN7SK,455.747883,-4.375801,0.553297,-7.908591,2.220446e-16,2.171152e-12,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
ENSG00000202198,7SK,303.688625,-3.769521,0.527742,-7.142735,2.220446e-16,2.171152e-12,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
ENSG00000202058,RN7SKP80,14.951145,-5.132970,0.858145,-5.981471,3.161915e-13,2.061147e-09,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
ENSG00000269900,ENSG00000269900,43.450875,-2.420166,0.448516,-5.395944,4.900302e-11,2.395758e-07,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
ENSG00000081237,PTPRC,27.428664,-5.929187,1.121367,-5.287462,1.180884e-10,4.618675e-07,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos


## <center> Plots

#### **Volcano plot**

In [6]:
# QC plot: volcano plot with decoupler

volcano_tab = deg.dropna(subset=["log2FoldChange", "padj"]).copy()
volcano_tab = volcano_tab.set_index("genesymbol", drop=False)

fig = dc.pl.volcano(
    data=volcano_tab,
    x="log2FoldChange",
    y="padj",
    top=20,
    thr_stat=0.5,
    thr_sign=0.05,
    figsize=(7, 6),
    dpi=120,
    return_fig=True,
)
fig.axes[0].set_title("HR_case__IFNG_pos vs HR_control__IFNG_pos")
fig.savefig(out_plots / "volcano.png", bbox_inches="tight")
fig.savefig(out_plots / "volcano.pdf", bbox_inches="tight")

plt.close(fig)


#### **Top DEGs**

As boxplots

In [7]:
topn=20
## subset adata to samples in DEA
select_grp=['HR_control__IFNG_pos','HR_case__IFNG_pos']
tmp=adata[adata.obs['grp'].isin(select_grp)].copy()
tmp.obs['grp']=tmp.obs['grp'].astype('category').cat.reorder_categories(select_grp,ordered=True)
## palette
pal = [c for c in sns.color_palette('Oranges',n_colors=2).as_hex()]
tmp.uns['grp_colors']=pal
## get topn genes in comparison
keys=deg.sort_values('padj',ascending=True)
keys=keys[keys['padj']<0.05]
keys=keys.head(topn)['genesymbol'].tolist()

In [8]:
## get table with norm counts and metadata in long format
ptab = pd.DataFrame(tmp.layers['cts.deseq2_norm'], index=tmp.obs.index,columns=tmp.var.index)
ptab = (
    ptab[keys].reset_index().
    melt(id_vars=['sample_id'],var_name='genesymbol',value_name='expr'))
### add grp from adata
ptab['grp']=ptab['sample_id'].map( tmp.obs['grp'].to_dict() )
ptab['grp']=['HR case, IFNG+' if grp=='HR_case__IFNG_pos' else 'HR ctrl, IFNG+' for grp in ptab['grp']]
ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR ctrl, IFNG+','HR case, IFNG+'],ordered=True)
#ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(tmp.obs['grp'].cat.categories,ordered=True)
## add stat and padj
cols=['stat','padj']
for col in cols:
    col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
    ptab[col]=ptab['genesymbol'].map(col_dict)
### plot title (ordered by significance)
#ptab=ptab.sort_values(by='padj',ascending=True)
ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

ptab.head()

,sample_id,genesymbol,expr,grp,stat,padj,p_title
0,SD2,RN7SK,38.077610,"HR case, IFNG+",-7.908591,2.17e-12,RN7SK\nStat: -7.91\nPadj: 2.17e-12
1,SD4,RN7SK,15.434112,"HR case, IFNG+",-7.908591,2.17e-12,RN7SK\nStat: -7.91\nPadj: 2.17e-12
2,SD6,RN7SK,29.630081,"HR case, IFNG+",-7.908591,2.17e-12,RN7SK\nStat: -7.91\nPadj: 2.17e-12
3,SD8,RN7SK,37.177065,"HR case, IFNG+",-7.908591,2.17e-12,RN7SK\nStat: -7.91\nPadj: 2.17e-12
4,SD10,RN7SK,51.782174,"HR case, IFNG+",-7.908591,2.17e-12,RN7SK\nStat: -7.91\nPadj: 2.17e-12


In [9]:
# as boxplot


## palette
pal = {grp:color for grp,color in zip(ptab['grp'].cat.categories,sns.color_palette('Oranges',n_colors=2).as_hex())}

plots_per_col=8
ncols=math.ceil(len(keys)/plots_per_col)
w=2.3*plots_per_col if plots_per_col>=4 else 6
h=4*ncols

contrast='grp'
p=(
    p9.ggplot(ptab, p9.aes(x=contrast, y='expr', group=contrast)) +
    p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.3,show_legend=False) +
    p9.geom_jitter(p9.aes(fill=contrast), color='black', size=3, width=0.1,alpha=1.0,show_legend=True) + 
    p9.scale_fill_manual(pal) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
    p9.theme_classic() +
    p9.labs(title="Expression of selected genes") +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text_x=p9.element_blank(), 
        legend_text=p9.element_text(size=12),
        figure_size=(w, h),
    ) + 
    p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
)
p.save(out_plots / "DEA_boxplot_01.png", width=w, height=h, limitsize=False)
p.save(out_plots / "DEA_boxplot_01.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/DEA_boxplot_01.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/DEA_boxplot_01.pdf


#### **Prev selected genes: Expression**

NOTE: these were the ones originally plotted in the manuscript figure

- TODO: add all groups

In [10]:
## subset adata to samples in DEA
tmp=adata.copy() # no change since we plot all samples
## get topn genes in comparison
keys=[
    'APOL1', 'APOL6', 'IRF1', 'CCL2', 'CXCL9', 'CXCL10', 
    'ICAM1', 'IDO1', 'PDCD1LG2', 'GBP1', 'PSMB9', 'CCDC68', 
    'CT5S', 'TMEM140', 'TRIM21', 'STAT1', 'STAT2', 'STAT3', 
    'IRF2', 'IRF8', 'IRF9', 'CXCL11']
keys=[k for k in keys if k in tmp.var_names]

As boxplots

In [11]:
## get table with norm counts and metadata in long format
ptab = pd.DataFrame(tmp.layers['cts.deseq2_norm'], index=tmp.obs.index,columns=tmp.var.index)
ptab = (
    ptab[keys].reset_index().
    melt(id_vars=['sample_id'],var_name='genesymbol',value_name='expr'))
### add grp from adata
ptab['grp']=ptab['sample_id'].map( tmp.obs['grp'].to_dict() )
col_dict={
    'HR_case__IFNG_neg':'HR case, IFNG-', 'HR_case__IFNG_pos':'HR case, IFNG+', 
    'HR_control__IFNG_neg':'HR control, IFNG-','HR_control__IFNG_pos':'HR control, IFNG+'}
ptab['grp']=ptab['grp'].map(col_dict)
ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG-','HR case, IFNG+','HR control, IFNG-','HR control, IFNG+'],ordered=True)
## add stat and padj
cols=['stat','padj']
for col in cols:
    col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
    ptab[col]=ptab['genesymbol'].map(col_dict)
### plot title (ordered by significance)
ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

ptab.head()

,sample_id,genesymbol,expr,grp,stat,padj,p_title
0,SD1,APOL1,14.334884,"HR case, IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01
1,SD3,APOL1,5.123411,"HR case, IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01
2,SD5,APOL1,18.323577,"HR case, IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01
3,SD7,APOL1,4.891112,"HR case, IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01
4,SD9,APOL1,40.231772,"HR case, IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01


In [12]:
# as boxplot


## palette
oranges=[c for c in sns.color_palette('Oranges',n_colors=2).as_hex()] # HR case
blues=[c for c in sns.color_palette('Blues',n_colors=2).as_hex()] # HR control
pal = {grp:color for grp,color in zip(ptab['grp'].cat.categories,oranges+blues)}

plots_per_col=8
ncols=math.ceil(len(keys)/plots_per_col)
w=2.3*plots_per_col if plots_per_col>=4 else 6
h=4*ncols

contrast='grp'
p=(
    p9.ggplot(ptab, p9.aes(x=contrast, y='expr', group=contrast)) +
    p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.3,show_legend=False) +
    p9.geom_jitter(p9.aes(fill=contrast), color='black', size=3, width=0.1,alpha=1.0,show_legend=True) + 
    p9.scale_fill_manual(pal) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
    p9.theme_classic() +
    p9.labs(title="Expression of selected genes") +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text_x=p9.element_blank(), 
        legend_text=p9.element_text(size=12),
        figure_size=(w, h),
    ) + 
    p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
)
p.save(out_plots / "Fig5g_boxplot_01.png", width=w, height=h, limitsize=False)
p.save(out_plots / "Fig5g_boxplot_01.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/Fig5g_boxplot_01.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/Fig5g_boxplot_01.pdf


#### **Prev selected genes: Fold change**

NOTE: these were the ones originally plotted in the manuscript figure

In [13]:
## subset adata to samples in DEA
tmp=adata.copy() # no change since we plot all samples
## get topn genes in comparison
keys=[
    'APOL1', 'APOL6', 'IRF1', 'CCL2', 'CXCL9', 'CXCL10', 
    'ICAM1', 'IDO1', 'PDCD1LG2', 'GBP1', 'PSMB9', 'CCDC68', 
    'CT5S', 'TMEM140', 'TRIM21', 'STAT1', 'STAT2', 'STAT3', 
    'IRF2', 'IRF8', 'IRF9', 'CXCL11']
keys=[k for k in keys if k in tmp.var_names]

As boxplots

In [ ]:
# get fold change
## by individualID, IFNG+ / IFNG-

## get expression
ptab = pd.DataFrame(tmp.layers['cts.deseq2_norm'], index=tmp.obs.index,columns=tmp.var.index)
ptab = (
    ptab[keys].reset_index().
    melt(id_vars=['sample_id'],var_name='genesymbol',value_name='expr'))
## add individualID
col_dict=tmp.obs['individualID'].to_dict()
ptab['individualID']=ptab['sample_id'].map(col_dict)
## add other columns
cols=['condition_group']
for col in cols:
    col_dict=tmp.obs[col].to_dict()
    ptab[col]=ptab['sample_id'].map(col_dict)
## get fold change
ptab = (
    ptab
    .groupby(["individualID", "genesymbol", "condition_group"], observed=True)["expr"]
    .mean()
    .reset_index()
    .pivot(index=["individualID", "genesymbol"], columns="condition_group", values="expr")
    .reset_index()
)
eps = 1e-6
ptab["fold_change"] = (ptab["IFNG_pos"] + eps) / (ptab["IFNG_neg"] + eps)
ptab["log2_fold_change"] = np.log2(ptab["fold_change"])

ptab.head()

condition_group,individualID,genesymbol,IFNG_neg,IFNG_pos,fold_change,log2_fold_change
0,individual_01,APOL1,14.334884,1782.217875,124.327326,6.958000
1,individual_01,APOL6,25.598007,1749.712598,68.353468,6.094943
2,individual_01,CCDC68,69.626578,78.012664,1.120444,0.164070
3,individual_01,CCL2,2445.121586,3906.205514,1.597551,0.675862
4,individual_01,CXCL10,33.789369,6323.669364,187.149670,7.548049


In [20]:
# prepare table for plotting

### add disease_group
col_dict=tmp.obs.set_index('individualID')['disease_group'].to_dict()
ptab['grp']=ptab['individualID'].map( col_dict )
col_dict={'HR_case':'HR case, IFNG+ vs IFNG-', 'HR_control':'HR control, IFNG+ vs IFNG-'}
ptab['grp']=ptab['grp'].map(col_dict)
ptab['grp']=ptab['grp'].astype('category').cat.reorder_categories(['HR case, IFNG+ vs IFNG-', 'HR control, IFNG+ vs IFNG-'],ordered=True)
## add stat and padj
cols=['stat','padj']
for col in cols:
    col_dict={x:y for x,y in zip(deg['genesymbol'],deg[col])}
    ptab[col]=ptab['genesymbol'].map(col_dict)
### plot title (ordered by significance)
ptab['padj']=['{:.2e}'.format(x) for x in ptab['padj']]
ptab['p_title']=[f"{a}\nStat: {round(float(b),2)}\nPadj: {c}" for a,b,c in zip(ptab['genesymbol'],ptab['stat'],ptab['padj'])]
ptab['p_title']=ptab['p_title'].astype('category').cat.reorder_categories(ptab['p_title'].unique(), ordered=True)

ptab.head()

condition_group,individualID,genesymbol,IFNG_neg,IFNG_pos,fold_change,log2_fold_change,grp,stat,padj,p_title
0,individual_01,APOL1,14.334884,1782.217875,124.327326,6.958000,"HR case, IFNG+ vs IFNG-",-0.296456,9.99e-01,APOL1\nStat: -0.3\nPadj: 9.99e-01
1,individual_01,APOL6,25.598007,1749.712598,68.353468,6.094943,"HR case, IFNG+ vs IFNG-",-0.037716,9.99e-01,APOL6\nStat: -0.04\nPadj: 9.99e-01
2,individual_01,CCDC68,69.626578,78.012664,1.120444,0.164070,"HR case, IFNG+ vs IFNG-",0.423266,9.94e-01,CCDC68\nStat: 0.42\nPadj: 9.94e-01
3,individual_01,CCL2,2445.121586,3906.205514,1.597551,0.675862,"HR case, IFNG+ vs IFNG-",-0.665791,9.58e-01,CCL2\nStat: -0.67\nPadj: 9.58e-01
4,individual_01,CXCL10,33.789369,6323.669364,187.149670,7.548049,"HR case, IFNG+ vs IFNG-",0.303272,9.99e-01,CXCL10\nStat: 0.3\nPadj: 9.99e-01


In [ ]:
# as boxplot


## palette
pal = [c for c in sns.color_palette('tab10',n_colors=2).as_hex()] # blue, orange
pal = pal[::-1]
pal = {cat:color for cat,color in zip(ptab['grp'].cat.categories,pal)}

plots_per_col=8
ncols=math.ceil(len(keys)/plots_per_col)
w=2.3*plots_per_col if plots_per_col>=4 else 6
h=4*ncols

contrast='grp'
p=(
    p9.ggplot(ptab, p9.aes(x=contrast, y='log2_fold_change', group=contrast)) +
    p9.geom_boxplot(outlier_alpha=0,size=0.8,width=0.3,show_legend=False) +
    p9.geom_jitter(p9.aes(fill=contrast), color='black', size=3, width=0.1,alpha=1.0,show_legend=True) + 
    p9.scale_fill_manual(pal) +
    p9.guides(fill=p9.guide_legend(override_aes={'size':6})) +
    p9.theme_classic() +
    p9.labs(title="Log2 fold change of selected genes",y='Log2FC',x='disease group') +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=14), 
        strip_text=p9.element_text(size=12), 
        axis_title=p9.element_text(size=14), 
        axis_text_x=p9.element_blank(), 
        legend_text=p9.element_text(size=12),
        figure_size=(w, h),
    ) + 
    p9.facet_wrap('p_title', scales = 'free', ncol=plots_per_col) 
)
p.save(out_plots / "Fig5g_boxplot_02.png", width=w, height=h, limitsize=False)
p.save(out_plots / "Fig5g_boxplot_02.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/Fig5g_boxplot_02.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 18.4 x 12 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/plot_DEA/Fig5g_boxplot_02.pdf
